# 第 32 课｜从零实现 LoRA
## 冻结基座、低秩更新、梯度、保存重载与合并

本课从真实小模型的量化评测回到一个可完全检查的小模块：**基座不更新，只训练两张小矩阵，为什么输出仍然能改变？**

我们用 PyTorch 从零实现 LoRA Linear，验证数学、梯度与文件契约，再完成已知秩的合成训练。最后把实现映射到真实模型的 `target_modules` 概念，为后续 PEFT 与 SFT 做准备。

**本 Notebook 自包含；同名 `.py` 也可独立运行。只需要 PyTorch 和 Python 标准库，不下载模型、数据或额外包。**

**边界：** 本课基座是固定随机矩阵，不是预训练语言模型；合成教师刻意使用秩为2的更新。实验检验机制，不代表 LoRA 的真实语言效果或相对全量微调的排名。FP32/FP64 稠密基座，未实现 QLoRA、混合精度或性能基准。

默认运行会在当前工作目录生成 `lesson_32_outputs`，重复运行覆盖同名生成文件，不改动前几课文件。

## 0. 学习方法与本课目标

先理解一个 Linear 的输入输出，再看多层梯度路径。长的“完整实现”单元默认折叠：先运行，随后结合讲解展开阅读；它们不是需要记忆的模板。

完成后应当能够解释：

- $W_0$、$A$、$B$ 的形状，为什么前向分支不用显式生成完整 $\Delta W$。
- 为什么 B 全零、A 随机时首步只有 B 的梯度通常非零；为什么二者都为零会卡住。
- 冻结参数与 `no_grad()` 的区别，以及如何检查实际上只训练了哪些参数。
- adapter 保存了什么，为什么仍需要正确的基座；何时可以导出合并权重。

先修：自动求导、Linear、优化器、矩阵秩/形状、模型保存。**不需要先安装 PEFT。**

In [1]:
from __future__ import annotations

# %% 01. Imports, contracts, reproducibility
import argparse
import copy
import hashlib
import json
import math
from pathlib import Path
import platform
import tempfile
from typing import Callable

import torch
from torch import nn
from torch.nn import functional as F

SOURCES = {
    "lora_paper": "https://arxiv.org/html/2106.09685v2",
    "original_linear": "https://github.com/microsoft/LoRA/blob/main/loralib/layers.py",
    "peft_lora": "https://huggingface.co/docs/peft/package_reference/lora",
    "autograd": "https://docs.pytorch.org/docs/2.10/notes/autograd.html",
    "linear": "https://docs.pytorch.org/docs/2.10/generated/torch.nn.Linear.html",
    "peft_checkpoint": "https://huggingface.co/docs/peft/developer_guides/checkpoint",
    "serialization": "https://docs.pytorch.org/docs/2.10/notes/serialization.html",
    "qlora": "https://arxiv.org/abs/2305.14314",
}
FORMAT = "lesson32-linear-lora-v1"
SEED = 3201


def positive_int(x, name):
    if type(x) is not int or x <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受 bool")


def finite_positive(x, name):
    if isinstance(x, bool) or not isinstance(x, (int, float)) or not math.isfinite(x) or x <= 0:
        raise ValueError(f"{name} 必须是有限正数")


def validate_lora(rank, alpha, dropout, in_features, out_features):
    positive_int(rank, "rank")
    positive_int(in_features, "in_features")
    positive_int(out_features, "out_features")
    # 为低秩教学作出的限制，不是说别的库绝对禁止更大的r。
    if rank > min(in_features, out_features):
        raise ValueError("本课 rank 不超过 min(in_features, out_features)")
    finite_positive(alpha, "alpha")
    if isinstance(dropout, bool) or not isinstance(dropout, (float, int)) or not 0 <= dropout < 1:
        raise ValueError("dropout 必须在 [0,1) 内")


def cpu_state(module):
    return {name: t.detach().cpu().clone() for name, t in module.state_dict().items()}


def state_sha256(state):
    """本课精确基座指纹；包含形状、dtype、名字、原始字节。不是跨精度兼容器。"""
    h = hashlib.sha256()
    for name, value in sorted(state.items()):
        t = value.detach().cpu().contiguous()
        h.update(f"{name}|{tuple(t.shape)}|{t.dtype}|".encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()


def choose_device(name):
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name not in {"cpu", "cuda"}:
        raise ValueError("device 必须是 cpu/cuda/auto")
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("显式请求 CUDA，但当前 PyTorch 无可用 CUDA；不会静默回退")
    return torch.device(name)


def seeded_linear(in_features=16, out_features=12, *, bias=True, seed=SEED,
                  dtype=torch.float32, device="cpu"):
    # fork_rng 避免初始化破坏调用者的CPU RNG；随后统一迁移设备。
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        layer = nn.Linear(in_features, out_features, bias=bias, dtype=dtype)
    return layer.to(device)


def trainable_parameters(module):
    result = [p for p in module.parameters() if p.requires_grad]
    if not result:
        raise ValueError("没有可训练参数：请检查冻结/注入是否正确")
    return result


def parameter_report(module):
    rows = [{"name": n, "shape": list(p.shape), "trainable": p.requires_grad,
             "numel": p.numel(), "dtype": str(p.dtype),
             "bytes": p.numel() * p.element_size()}
            for n, p in module.named_parameters()]
    return {"total": sum(r["numel"] for r in rows),
            "trainable": sum(r["numel"] for r in rows if r["trainable"]), "rows": rows}

In [2]:
# 仅本课运行时使用单CPU线程，避免小矩阵受到线程调度影响。
torch.set_num_threads(1)
DEVICE = "cpu"           # 可改为 "cuda" 或 "auto"；本次保存的输出来自CPU。
STEPS = 400
LR = 0.03
RUN_TRAINING = True
OUTPUT_DIR = Path("lesson_32_outputs")
device = choose_device(DEVICE)
print({"python": platform.python_version(), "torch": str(torch.__version__),
       "actual_device": str(device), "cuda_available": torch.cuda.is_available()})

{'python': '3.13.5', 'torch': '2.10.0+cpu', 'actual_device': 'cpu', 'cuda_available': False}


## 1. LoRA 约束的是“更新”，不是把原权重压缩掉

对一个线性层，保留原权重 $W_0$，学习增量：

$$W_{\mathrm{eff}}=W_0+\Delta W,\qquad \Delta W=\frac{\alpha}{r}BA$$

其中 $W_0$ 冻结，A、B 可训练。这里采用标准 LoRA 的 `alpha/r` 缩放，不涉及 rsLoRA、DoRA 等变体。[1][2]

PyTorch Linear 的权重为 `[out_features, in_features]`，输入行向量计算 $XW^\top+b$。[3]

| 对象 | 形状 |
|---|---|
| 输入 X | `[B,T,in]`，也支持 `[N,in]` 或 `[in]` |
| 基座 W0 | `[out,in]` |
| A | `[r,in]` |
| B | `[out,r]` |
| BA | `[out,in]` |
| 输出 | `[B,T,out]` |

**W0 可以是满秩；只要求增量的秩不超过 r。整个多层非线性网络也不能因此称为“秩为r的模型”。**

In [3]:
# 先用没有偏置的手写数字，检查两个乘法路径。
X = torch.tensor([[1., 2., 3., 4.]], dtype=torch.float64)
W0 = torch.zeros(3, 4, dtype=torch.float64)
A = torch.tensor([[1., 0., 0., 0.], [0., 1., 0., 0.]], dtype=torch.float64)
B = torch.tensor([[1., 0.], [0., 1.], [1., 1.]], dtype=torch.float64)
rank, alpha = 2, 4.0
s = alpha / rank
branch = X @ W0.T + s * ((X @ A.T) @ B.T)
dense = X @ (W0 + s * (B @ A)).T
print("A、B、BA形状：", A.shape, B.shape, (B @ A).shape)
print("分支输出：", branch)
print("稠密输出：", dense)
torch.testing.assert_close(branch, dense)

A、B、BA形状： torch.Size([2, 4]) torch.Size([3, 2]) torch.Size([3, 4])
分支输出： tensor([[2., 4., 6.]], dtype=torch.float64)
稠密输出： tensor([[2., 4., 6.]], dtype=torch.float64)


### 1.1 不是冻结矩阵分解，也不是剪枝

LoRA 不先把 W0 分解成两个低秩矩阵来替代它，也不直接删除原权重。它添加一条**并行的可训练修正分支**。[1]

前向按以下顺序计算：

```python
base_output = base(x)
low_rank = F.linear(dropout(x), A)
delta_output = F.linear(low_rank, B)
return base_output + (alpha / r) * delta_output
```

先把 `in` 投影到 r，再映射到 `out`。`F.linear(x, A)` 已经处理权重转置，不要再次把 A 转置进去。训练时不需要先物化大矩阵 BA。

本课的 Dropout 只作用于 LoRA 分支，不作用于原基座输出。偏置 b 保留但冻结。

In [4]:
# %% 02. The complete LoRA Linear implementation
class LoRALinear(nn.Module):
    """普通稠密 nn.Linear 的LoRA封装，权重方向为 [out,in]。

    复制基座，不与调用者共享参数；基座权重/偏置冻结且清除旧grad。
    A [r,in] 随机，B [out,r] 全零；标准scaling=alpha/r。
    Dropout只作用在LoRA分支的输入。只支持FP32/FP64教学训练。
    不支持量化Linear、混合精度、权重共享、就地merge或adapter切换。
    """
    def __init__(self, base: nn.Linear, rank=2, alpha=2.0, dropout=0.0, seed=SEED):
        super().__init__()
        if type(base) is not nn.Linear:
            raise TypeError("本课只接受原生 nn.Linear，不接受量化层或已有LoRA层")
        validate_lora(rank, alpha, dropout, base.in_features, base.out_features)
        if base.weight.dtype not in {torch.float32, torch.float64}:
            raise ValueError("本课仅支持FP32/FP64；低精度/量化集成留给后续课程")
        if base.bias is not None and (base.bias.dtype != base.weight.dtype or base.bias.device != base.weight.device):
            raise ValueError("基座权重和偏置的dtype/device必须一致")
        if any(not bool(torch.isfinite(p).all()) for p in base.parameters()):
            raise ValueError("基座包含非有限参数")
        self.base = copy.deepcopy(base)
        self.base.requires_grad_(False)
        for p in self.base.parameters():
            p.grad = None
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        self.alpha = float(alpha)
        self.scaling = self.alpha / self.rank
        self.dropout_p = float(dropout)
        self.dropout = nn.Dropout(self.dropout_p)
        # 等价于Kaiming uniform(a=sqrt(5))的边界；本地generator不污染全局RNG。
        generator = torch.Generator(device="cpu").manual_seed(seed)
        initial_a = torch.empty(rank, self.in_features, dtype=base.weight.dtype)
        initial_a.uniform_(-1 / math.sqrt(self.in_features), 1 / math.sqrt(self.in_features), generator=generator)
        self.lora_A = nn.Parameter(initial_a.to(base.weight.device))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank,
                                              dtype=base.weight.dtype, device=base.weight.device))
        self.train(base.training)

    def forward(self, x):
        # 冻结参数不代表用no_grad包住基座：x可能来自更早的可训练层。
        base_output = self.base(x)
        low_rank = F.linear(self.dropout(x), self.lora_A)
        delta_output = F.linear(low_rank, self.lora_B)
        return base_output + self.scaling * delta_output

    def delta_weight(self):
        """仅供检查/合并；训练forward不物化完整ΔW。"""
        return self.scaling * (self.lora_B @ self.lora_A)

    def merged_linear(self):
        """返回独立的普通稠密层，不改变原模型。仅允许eval状态。"""
        if self.training:
            raise RuntimeError("先调用 eval() 再合并；活动Dropout不能静态合并")
        with torch.no_grad():
            weight = self.base.weight + self.delta_weight()
            if not bool(torch.isfinite(weight).all()):
                raise ValueError("合并结果非有限，拒绝导出")
            merged = copy.deepcopy(self.base)
            merged.weight.copy_(weight)
        merged.requires_grad_(False)
        merged.eval()
        return merged

In [5]:
layer = LoRALinear(seeded_linear(5, 4, dtype=torch.float64), rank=2, alpha=4.0)
x = torch.arange(30, dtype=torch.float64).reshape(2, 3, 5) / 10
print("输入/输出：", x.shape, layer(x).shape)
print("初始与基座差异：", (layer(x) - layer.base(x)).abs().max().item())
for row in parameter_report(layer)["rows"]:
    print(row["name"], row["shape"], "trainable=", row["trainable"])

输入/输出： torch.Size([2, 3, 5]) torch.Size([2, 3, 4])
初始与基座差异： 0.0
lora_A [2, 5] trainable= True
lora_B [4, 2] trainable= True
base.weight [4, 5] trainable= False
base.bias [4] trainable= False


## 2. 参数到底减少了多少？

单个无偏置权重的全量更新参数数为 $d_{out}d_{in}$；LoRA 的可训练参数为：

$$r(d_{in}+d_{out})$$

**不是“整个模型只剩这些参数”。基座仍然存在并参与计算。**

例如4096×4096的单个矩阵，r=8：全量权重为16,777,216个，A/B为65,536个，可训练数约为全量的0.390625%。这是公式核算，不是实际GPU显存测量。

反例：对3×4的小矩阵，r=2会引入 `2×(3+4)=14` 个参数，超过原矩阵的12个。只有r足够小，才有参数优势；秩越大也不是越划算。

In [6]:
# %% 05. Mechanism probes: gradients, rank, scaling, dropout, merge
def geometry_demo():
    dtype = torch.float64
    base = seeded_linear(4, 3, dtype=dtype)
    layer = LoRALinear(base, rank=2, alpha=6.0).eval()
    with torch.no_grad():
        layer.lora_A.copy_(torch.tensor([[1., 0., 2., -1.], [0., 1., 1., 2.]], dtype=dtype))
        layer.lora_B.copy_(torch.tensor([[1., 2.], [0., -1.], [2., 1.]], dtype=dtype))
    x = torch.arange(24, dtype=dtype).reshape(2, 3, 4) / 10
    actual = layer(x)
    dense = F.linear(x, base.weight + layer.delta_weight(), base.bias)
    return {"input_shape": list(x.shape), "A_shape": list(layer.lora_A.shape),
            "B_shape": list(layer.lora_B.shape), "delta_shape": list(layer.delta_weight().shape),
            "output_shape": list(actual.shape), "scaling": layer.scaling,
            "delta_rank": int(torch.linalg.matrix_rank(layer.delta_weight()).item()),
            "dense_max_error": (actual-dense).abs().max().item(),
            "trainable": parameter_report(layer)["trainable"],
            "full_weight_numel": base.weight.numel()}


def initialization_demo():
    layer = LoRALinear(seeded_linear(6, 4, dtype=torch.float64), rank=2, alpha=4.0)
    gen = torch.Generator().manual_seed(3210)
    x = torch.randn(8, 6, generator=gen, dtype=torch.float64)
    target = torch.randn(8, 4, generator=gen, dtype=torch.float64)
    optimizer = torch.optim.SGD(trainable_parameters(layer), lr=0.1)
    first_output_error = (layer(x)-layer.base(x)).abs().max().item()
    records = []
    for step in range(2):
        optimizer.zero_grad(set_to_none=True)
        loss = F.mse_loss(layer(x), target)
        loss.backward()
        records.append({"backward": step+1, "A_grad_norm": layer.lora_A.grad.norm().item(),
                        "B_grad_norm": layer.lora_B.grad.norm().item(),
                        "base_grad_none": all(p.grad is None for p in layer.base.parameters())})
        optimizer.step()
    dead = LoRALinear(seeded_linear(6, 4, dtype=torch.float64), rank=2)
    with torch.no_grad():
        dead.lora_A.zero_()
        dead.lora_B.zero_()
    F.mse_loss(dead(x), target).backward()
    return {"initial_output_max_error": first_output_error, "steps": records,
            "both_zero_A_grad_norm": dead.lora_A.grad.norm().item(),
            "both_zero_B_grad_norm": dead.lora_B.grad.norm().item()}


def frozen_path_demo():
    """冻结第二层仍须把梯度传给更早的LoRA；显式构造B非零以观察路径。"""
    model = inject_lora(TinyTwoLayer(), ["proj_in"], rank=2, alpha=2.0)
    with torch.no_grad():
        model.proj_in.lora_B.fill_(0.1)
    x = torch.arange(30, dtype=torch.float64).reshape(5, 6) / 30
    target = torch.ones(5, 4, dtype=torch.float64)
    loss = F.mse_loss(model(x), target)
    loss.backward()
    correct_norm = model.proj_in.lora_A.grad.norm().item()
    hidden = F.gelu(model.proj_in(x))
    with torch.no_grad():
        bad_output = model.proj_out(hidden)
    return {"trainable_names": [n for n, p in model.named_parameters() if p.requires_grad],
            "upstream_A_grad_norm": correct_norm,
            "frozen_output_grad_none": all(p.grad is None for p in model.proj_out.parameters()),
            "no_grad_output_requires_grad": bad_output.requires_grad,
            "wrong_path_cannot_backpropagate": not bad_output.requires_grad}


def merge_demo():
    layer = LoRALinear(seeded_linear(5, 3, dtype=torch.float64), 2, 6.0, dropout=0.25).eval()
    with torch.no_grad():
        layer.lora_B.fill_(0.2)
    x = torch.arange(20, dtype=torch.float64).reshape(4, 5) / 10
    before = state_sha256(cpu_state(layer))
    merged = layer.merged_linear()
    merged2 = layer.merged_linear()
    correct = layer(x)
    wrong = merged(x) + F.linear(F.linear(x, layer.lora_A), layer.lora_B)*layer.scaling
    return {"merge_error": (correct-merged(x)).abs().max().item(),
            "source_unchanged": state_sha256(cpu_state(layer)) == before,
            "repeated_export_error": (merged(x)-merged2(x)).abs().max().item(),
            "double_count_error": (correct-wrong).abs().max().item(),
            "merged_has_adapter": any("lora_" in n for n, _ in merged.named_parameters())}


def capacity_demo():
    # 单个4096×4096投影、r=8；只做整数计算，不实际分配大矩阵。
    din = dout = 4096
    r = 8
    full = din*dout
    adapter = r*(din+dout)
    return {"in_features": din, "out_features": dout, "rank": r,
            "full_weight_parameters": full, "lora_parameters": adapter,
            "trainable_fraction_vs_full_weight": adapter/full,
            "base_fp32_bytes": 4*full, "adapter_fp32_bytes": 4*adapter,
            "adapter_grad_fp32_bytes": 4*adapter,
            "adam_moments_fp32_bytes": 8*adapter,
            "note": "仅元素载荷；未计激活、临时张量、优化器步数和分配器。基座仍需保存。"}

In [7]:
print(json.dumps(capacity_demo(), ensure_ascii=False, indent=2))
geometry = geometry_demo()
print("小矩阵：", json.dumps(geometry, ensure_ascii=False, indent=2))

{
  "in_features": 4096,
  "out_features": 4096,
  "rank": 8,
  "full_weight_parameters": 16777216,
  "lora_parameters": 65536,
  "trainable_fraction_vs_full_weight": 0.00390625,
  "base_fp32_bytes": 67108864,
  "adapter_fp32_bytes": 262144,
  "adapter_grad_fp32_bytes": 262144,
  "adam_moments_fp32_bytes": 524288,
  "note": "仅元素载荷；未计激活、临时张量、优化器步数和分配器。基座仍需保存。"
}
小矩阵： {
  "input_shape": [
    2,
    3,
    4
  ],
  "A_shape": [
    2,
    4
  ],
  "B_shape": [
    3,
    2
  ],
  "delta_shape": [
    3,
    4
  ],
  "output_shape": [
    2,
    3,
    3
  ],
  "scaling": 3.0,
  "delta_rank": 2,
  "dense_max_error": 1.0658141036401503e-14,
  "trainable": 14,
  "full_weight_numel": 12
}


## 3. 为什么把 B 初始化为零，而 A 不能也为零？

常见初始化是 **A 随机、B 全零**。因此初始增量 BA=0，模型输出与原基座相同。原论文使用随机高斯 A；本课使用等价于 Kaiming-uniform 的边界，与 PEFT 常见默认相近。[1][2]

为了看梯度，把 $s=\alpha/r$、$G=\partial L/\partial W_{eff}$ 代入链式法则（此处不含Dropout）：

$$\frac{\partial L}{\partial A}=sB^\top G,\qquad
\frac{\partial L}{\partial B}=sGA^\top$$

初始B=0，所以 A 的首步梯度为零。A随机时 B 的梯度一般可以非零（不是任何数据/损失下都保证非零）。B更新后，下一次A通常也能得到非零梯度。

**A的首步零梯度不是自动求导失效。** 但如果A、B都为零，那么这条双线性分支的两个梯度都为零，会卡住。

In [8]:
initialization = initialization_demo()
print("初始输出误差：", initialization["initial_output_max_error"])
for row in initialization["steps"]:
    print(row)
print("A/B都为零的梯度：", initialization["both_zero_A_grad_norm"],
      initialization["both_zero_B_grad_norm"])

初始输出误差： 0.0
{'backward': 1, 'A_grad_norm': 0.0, 'B_grad_norm': 1.646905011178786, 'base_grad_none': True}
{'backward': 2, 'A_grad_norm': 0.36702503583777313, 'B_grad_norm': 1.4736791341306146, 'base_grad_none': True}
A/B都为零的梯度： 0.0 0.0


### 3.1 alpha、rank 和学习率不是一个参数

- r 决定瓶颈宽度和参数数量，并给增量矩阵的秩设置上界。
- alpha/r 直接缩放前向增量，也通过链式法则影响梯度。
- 学习率控制优化器如何用梯度更新A/B。

改变alpha并不在所有初始化、优化器、正则化条件下都等价于改变学习率。改变r时是否固定alpha也必须写入实验协议。

本课的秩对照采用 **alpha=r，所有分支scaling=1**，避免一边改rank、一边无意改变显式增量比例。A的尺寸和优化问题仍然随rank变化，并不是一切条件完全相同。

In [9]:
base_demo = seeded_linear(5, 4, dtype=torch.float64)
scaled = LoRALinear(base_demo, 2, 6.0).eval()  # 特意选alpha/r=3，而不是1
with torch.no_grad():
    scaled.lora_B.fill_(0.2)
x_demo = torch.ones(2, 5, dtype=torch.float64)
branch_without_scale = F.linear(F.linear(x_demo, scaled.lora_A), scaled.lora_B)
actual_delta = scaled(x_demo) - scaled.base(x_demo)
print("scaling：", scaled.scaling)
print("遗漏缩放的最大误差：", (actual_delta-branch_without_scale).abs().max().item())
torch.testing.assert_close(actual_delta, 3 * branch_without_scale)

scaling： 3.0
遗漏缩放的最大误差： 0.7350446621044742


## 4. 冻结参数，不等于停止经过它的梯度

`requires_grad_(False)` 表示不要计算/累积该叶子参数的梯度。只要输入来自可训练计算，梯度仍可经过冻结的线性层传回更早的位置。[4]

例如：

```text
可训练LoRA第一层 → GELU → 冻结第二层 → loss
```

第二层不更新，但第一层仍需要从它收到误差信号。

**错误做法：** 把冻结第二层包进 `with torch.no_grad():`，会切断这条输入路径。类似地，对结果 `.detach()` 不是“省梯度内存”的通用替代。[4]

如果某个冻结前缀完全没有需要梯度的输入/参数，PyTorch可以不记录它的图；不能由此推广到已经混入可训练适配器的整个基座。

In [10]:
# %% 03. Small network: exact target names, frozen-base audit, atomic injection
class TinyTwoLayer(nn.Module):
    """只用于测试多层梯度连通性，不是Transformer/语言模型。"""
    def __init__(self, seed=SEED, dtype=torch.float64):
        super().__init__()
        self.proj_in = seeded_linear(6, 8, seed=seed, dtype=dtype)
        self.proj_out = seeded_linear(8, 4, seed=seed + 1, dtype=dtype)

    def forward(self, x):
        return self.proj_out(F.gelu(self.proj_in(x)))


def inject_lora(model, targets, rank=2, alpha=2.0, dropout=0.0):
    """返回副本；按完整路径匹配，不模糊替换。先验证全部目标，再改变副本。

    为避免把绑定权重意外拆开，本课拒绝共享模块/共享Parameter模型。
    真实模型的权重绑定、融合QKV、量化类等交给后续PEFT课处理。
    """
    if not isinstance(targets, (list, tuple)) or not targets or any(not isinstance(n, str) or not n for n in targets):
        raise ValueError("targets 必须是非空完整模块路径列表")
    if len(set(targets)) != len(targets):
        raise ValueError("targets 重复")
    if any(isinstance(m, LoRALinear) for m in model.modules()):
        raise ValueError("拒绝在已有LoRA模型上重复注入")
    module_refs = [id(m) for _, m in model.named_modules(remove_duplicate=False)]
    parameter_refs = [id(p) for _, p in model.named_parameters(remove_duplicate=False)]
    if len(module_refs) != len(set(module_refs)) or len(parameter_refs) != len(set(parameter_refs)):
        raise ValueError("本课不自动处理共享模块/绑定参数")
    modules = dict(model.named_modules())
    for name in targets:
        if name not in modules or type(modules[name]) is not nn.Linear:
            raise ValueError(f"目标 {name!r} 不存在，或不是原生 nn.Linear")
        validate_lora(rank, alpha, dropout, modules[name].in_features, modules[name].out_features)
    result = copy.deepcopy(model)
    result.requires_grad_(False)
    for p in result.parameters():
        p.grad = None
    for index, name in enumerate(targets):
        parent_name, _, attr = name.rpartition(".")
        parent = result.get_submodule(parent_name) if parent_name else result
        old = getattr(parent, attr)
        setattr(parent, attr, LoRALinear(old, rank, alpha, dropout, seed=SEED + index))
    allowed = {f"{name}.lora_{letter}" for name in targets for letter in ("A", "B")}
    actual = {n for n, p in result.named_parameters() if p.requires_grad}
    if actual != allowed:
        raise RuntimeError("可训练参数集合与注入计划不一致")
    return result

In [11]:
frozen_probe = frozen_path_demo()
print(json.dumps(frozen_probe, ensure_ascii=False, indent=2))

{
  "trainable_names": [
    "proj_in.lora_A",
    "proj_in.lora_B"
  ],
  "upstream_A_grad_norm": 0.05285943827560154,
  "frozen_output_grad_none": true,
  "no_grad_output_requires_grad": false,
  "wrong_path_cannot_backpropagate": true
}


## 5. 从一个层扩展到网络：先确定 target_modules

真实模型里，LoRA 可以加入选定的注意力或FFN投影。模块名称因架构而异；后续使用PEFT时需要核对实际命中的层。[2]

本课只演示两个普通Linear：`proj_in`、`proj_out`。注入函数返回副本，先验证全部名字，再冻结全部基座，仅把指定层的 A/B 设为可训练。**不存在的目标、重复目标、共享权重都明确拒绝**，不把0个命中当作成功。

这是精确完整路径匹配，不是PEFT所有匹配规则的实现；不支持真实模型中的量化类、融合QKV和绑定权重。

**优化器必须在注入之后创建**，否则旧优化器持有的可能仍是旧Parameter，根本不会更新新A/B。

In [12]:
original_network = TinyTwoLayer()
adapted_network = inject_lora(original_network, ["proj_in", "proj_out"], rank=2, alpha=4.0)
print("可训练参数：")
for name, parameter in adapted_network.named_parameters():
    if parameter.requires_grad:
        print(name, tuple(parameter.shape))
print("总数/可训练数：", parameter_report(adapted_network)["total"],
      parameter_report(adapted_network)["trainable"])
optimizer_demo = torch.optim.Adam(trainable_parameters(adapted_network), lr=0.01)
optimizer_ids = {id(p) for group in optimizer_demo.param_groups for p in group["params"]}
actual_ids = {id(p) for p in adapted_network.parameters() if p.requires_grad}
print("优化器恰好包含所有适配器参数：", optimizer_ids == actual_ids)

可训练参数：
proj_in.lora_A (2, 6)
proj_in.lora_B (8, 2)
proj_out.lora_A (2, 8)
proj_out.lora_B (4, 2)
总数/可训练数： 144 52
优化器恰好包含所有适配器参数： True


### 5.1 一次更新仍然是熟悉的训练循环

只训练A/B，不是绕过loss、反向传播和优化器。最可靠的核对包括：

1. 可训练参数名单精确匹配计划。
2. 训练后基座逐位不变，基座grad仍为None。
3. B能更新，第二步A也能更新；adapter输出确实改变。

保存“冻结前”残留的grad可能导致错误检查；本课封装会清除旧grad。创建优化器时也只传入 `requires_grad=True` 的参数。

In [13]:
step_layer = LoRALinear(seeded_linear(5, 4, dtype=torch.float64), 2, 4.0)
step_optimizer = torch.optim.SGD(trainable_parameters(step_layer), lr=0.1)
base_before = state_sha256(cpu_state(step_layer.base))
x_step = torch.arange(20, dtype=torch.float64).reshape(4, 5) / 20
y_step = torch.ones(4, 4, dtype=torch.float64)
for step in range(2):
    step_optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(step_layer(x_step), y_step)
    loss.backward()
    print("step", step+1, "loss", loss.item(), "A梯度范数", step_layer.lora_A.grad.norm().item())
    step_optimizer.step()
print("基座完全不变：", state_sha256(cpu_state(step_layer.base)) == base_before)
print("基座grad为空：", all(p.grad is None for p in step_layer.base.parameters()))

step 1 loss 2.002065078483839 A梯度范数 0.0
step 2 loss 1.6918725677877522 A梯度范数 0.5191320012298599
基座完全不变： True
基座grad为空： True


## 6. Dropout、eval 与安全合并

在评估模式，LoRA分支Dropout关闭，所以可以导出：

$$W_{merged}=W_0+\frac{\alpha}{r}BA$$

随后只调用一个普通Linear，偏置与基座保持一致。**这不会压缩整个权重矩阵**，导出仍是同形状的稠密权重。[1][2]

本课选择“返回独立副本”的合并方式，不就地反复加减原权重：

- 先 `eval()`，避免活动Dropout导致无法静态表示。
- 合并后使用返回的普通Linear，不再额外执行adapter分支。
- 重复调用导出函数得到相同结果，不会把ΔW加两次。
- 原LoRA层和源基座都保持不变。

本课不实现量化基座合并。量化权重直接加增量、反量化后合并、合并后再量化，是不同的工程路径，可能引入新的误差。

In [14]:
merge_result = merge_demo()
print(json.dumps(merge_result, ensure_ascii=False, indent=2))

{
  "merge_error": 4.440892098500626e-16,
  "source_unchanged": true,
  "repeated_export_error": 0.0,
  "double_count_error": 1.8151794476356136,
  "merged_has_adapter": false
}


## 7. Adapter 文件不是完整模型

PEFT风格的adapter保存通常只有新增参数与配置，需要相同的基座才能正确加载。[5]

本课教学格式只保存：

```text
A、B
in/out维度、rank、alpha、dropout、缩放约定、权重布局
基座参数SHA256指纹
```

我们故意让指纹包含精确字节与dtype；换成不同精度的基座也会拒绝。这是严格的教学契约，不是所有生产工具的通用规则。

**同形状不等于同一基座。** 改一个基座权重，adapter仍能参与矩阵乘法，但语义已经不再是原训练结果。真实模型还要匹配仓库版本、模块名、分词器与聊天模板。

下面的 `.pt` 是自定义教学格式，不能直接传给 `PeftModel.from_pretrained()`。只读取可信文件；`weights_only=True`也不是任意文件的安全保证。[6]

In [15]:
# %% 04. Exact-base adapter-only persistence (not a PEFT format)
def adapter_payload(layer):
    if not isinstance(layer, LoRALinear):
        raise TypeError("要求 LoRALinear")
    return {
        "format": FORMAT,
        "config": {"in_features": layer.in_features, "out_features": layer.out_features,
                   "bias": layer.base.bias is not None, "rank": layer.rank,
                   "alpha": layer.alpha, "dropout": layer.dropout_p,
                   "scaling_rule": "alpha/r", "weight_layout": "out,in"},
        "base_sha256": state_sha256(cpu_state(layer.base)),
        "state": {"lora_A": layer.lora_A.detach().cpu().clone(),
                  "lora_B": layer.lora_B.detach().cpu().clone()},
    }


def adapter_from_payload(base, payload):
    if not isinstance(payload, dict) or payload.get("format") != FORMAT:
        raise ValueError("不是本课adapter格式")
    cfg = payload.get("config")
    required = {"in_features", "out_features", "bias", "rank", "alpha", "dropout", "scaling_rule", "weight_layout"}
    if not isinstance(cfg, dict) or set(cfg) != required:
        raise ValueError("adapter配置缺失或包含不支持的字段")
    if cfg["scaling_rule"] != "alpha/r" or cfg["weight_layout"] != "out,in":
        raise ValueError("缩放规则/布局不兼容")
    if type(base) is not nn.Linear or type(cfg["bias"]) is not bool or (
        cfg["in_features"], cfg["out_features"], cfg["bias"]
    ) != (base.in_features, base.out_features, base.bias is not None):
        raise ValueError("adapter与基座结构不匹配")
    validate_lora(cfg["rank"], cfg["alpha"], cfg["dropout"], cfg["in_features"], cfg["out_features"])
    if payload.get("base_sha256") != state_sha256(cpu_state(base)):
        raise ValueError("基座指纹不匹配：同形状不代表同一权重/精度")
    state = payload.get("state")
    shapes = {"lora_A": (cfg["rank"], cfg["in_features"]),
              "lora_B": (cfg["out_features"], cfg["rank"])}
    if not isinstance(state, dict) or set(state) != set(shapes):
        raise ValueError("adapter参数字段必须只有 lora_A/lora_B")
    for name, shape in shapes.items():
        t = state[name]
        if not isinstance(t, torch.Tensor) or tuple(t.shape) != shape or t.dtype != base.weight.dtype:
            raise ValueError(f"{name} 的形状/dtype不匹配")
        if not bool(torch.isfinite(t).all()):
            raise ValueError(f"{name} 含非有限值")
    result = LoRALinear(base, cfg["rank"], cfg["alpha"], cfg["dropout"])
    with torch.no_grad():
        result.lora_A.copy_(state["lora_A"])
        result.lora_B.copy_(state["lora_B"])
    return result.eval()


def save_adapter(layer, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(adapter_payload(layer), path)


def load_adapter(base, path):
    # 只加载可信的本课文件；weights_only不意味着任意来源文件完全安全。
    payload = torch.load(path, map_location="cpu", weights_only=True)
    return adapter_from_payload(base, payload)


def load_dense_file(path, device="cpu"):
    """重建本课导出的独立稠密Linear；不会读取任意模型类。"""
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or payload.get("format") not in {"lesson32-base-v1", "lesson32-merged-v1"}:
        raise ValueError("不是本课稠密权重文件")
    din, dout, bias = payload.get("in_features"), payload.get("out_features"), payload.get("bias")
    positive_int(din, "in_features")
    positive_int(dout, "out_features")
    if type(bias) is not bool:
        raise ValueError("bias 必须是 bool")
    state = payload.get("state")
    if not isinstance(state, dict) or set(state) != ({"weight", "bias"} if bias else {"weight"}):
        raise ValueError("稠密文件参数字段错误")
    dtype = state["weight"].dtype
    if dtype not in {torch.float32, torch.float64}:
        raise ValueError("本课稠密文件仅支持FP32/FP64")
    for name, shape in [("weight", (dout, din))] + ([("bias", (dout,))] if bias else []):
        value = state[name]
        if not isinstance(value, torch.Tensor) or tuple(value.shape) != shape or value.dtype != dtype or not bool(torch.isfinite(value).all()):
            raise ValueError("稠密文件参数形状、精度或数值非法")
    result = seeded_linear(din, dout, bias=bias, dtype=dtype, device=device)
    result.load_state_dict(state, strict=True)
    return result.requires_grad_(False).eval()


def reload_saved_run(directory, device="cpu"):
    """完全从磁盘重建基座+adapter与合并模型；不训练，不写入源文件。"""
    directory = Path(directory)
    base = load_dense_file(directory / "base.pt", device)
    adapter = load_adapter(base, directory / "adapter_r2.pt")
    merged = load_dense_file(directory / "merged_r2.pt", device)
    x = torch.arange(48, dtype=base.weight.dtype, device=device).reshape(3, 16) / 48
    with torch.no_grad():
        error = (adapter(x) - merged(x)).abs().max().item()
        torch.testing.assert_close(adapter(x), merged(x), rtol=1e-5, atol=2e-6)
    return {"device":str(device), "adapter_trainable":parameter_report(adapter)["trainable"],
            "base_sha256":state_sha256(cpu_state(base)), "adapter_vs_merged_error":error,
            "note":"仅恢复推理；没有优化器、数据游标或RNG，不能精确续训"}

In [16]:
# 临时目录退出后自动删除；这里只验证适配器协议。
persist_layer = step_layer.eval()
with tempfile.TemporaryDirectory(prefix="lesson32-notebook-") as tmp:
    path = Path(tmp) / "adapter.pt"
    save_adapter(persist_layer, path)
    payload = torch.load(path, map_location="cpu", weights_only=True)
    restored = load_adapter(persist_layer.base, path)
    print("adapter张量字段：", list(payload["state"]))
    print("重载输出误差：", (restored(x_step)-persist_layer(x_step)).abs().max().item())
    wrong_base = copy.deepcopy(persist_layer.base)
    with torch.no_grad():
        wrong_base.weight[0, 0] += 0.01
    try:
        load_adapter(wrong_base, path)
    except ValueError as error:
        print("同形状错误基座被拒绝：", error)

adapter张量字段： ['lora_A', 'lora_B']
重载输出误差： 0.0
同形状错误基座被拒绝： 基座指纹不匹配：同形状不代表同一权重/精度


## 8. 完整训练：构造一个已知rank=2的目标

为了区别“秩不够”与“训练程序有bug”，构造：

$$W_* = W_0 + U\operatorname{diag}(2,1)V^\top$$

U、V各有两个正交列，所以目标增量的两个非零奇异值是2和1。输入维度16、输出维度12，偏置始终保持不变。

**W0是固定随机矩阵，不是预训练模型。** 这个任务刻意适合低秩学习；它证明实现能学到目标更新，不证明真实任务只需要rank=2。

训练输入是32个向量：正负 `sqrt(16)` 倍的标准基向量，满足：

$$\frac{1}{N}X^\top X=I$$

另外生成256个独立标准正态输入作为测试。只在全部固定步数结束后报告测试结果，不用测试集选择rank、学习率或轮次。

In [17]:
# %% 06. Known-rank teacher: controlled synthetic training, no language benchmark
def make_problem(device="cpu", dtype=torch.float32):
    """构造W*=W0+Δ*，Δ*奇异值为2和1，秩精确为2（容差内）。

    训练输入±sqrt(d)*I的二阶矩恰为I；测试为独立标准正态样本。
    因此训练输出MSE = ||W-W*||_F² / out_features。
    r=1最佳MSE=1/12；原始未适配MSE=(4+1)/12。
    此任务特意适合低秩更新，不是LoRA优于全量微调的证据。
    """
    din, dout = 16, 12
    base = seeded_linear(din, dout, seed=3200, dtype=dtype, device=device)
    generator = torch.Generator().manual_seed(3202)
    u, _ = torch.linalg.qr(torch.randn(dout, 2, generator=generator, dtype=torch.float64))
    v, _ = torch.linalg.qr(torch.randn(din, 2, generator=generator, dtype=torch.float64))
    delta = ((u * torch.tensor([2., 1.], dtype=torch.float64)) @ v.T).to(device=device, dtype=dtype)
    train_x = torch.cat([torch.eye(din, dtype=dtype), -torch.eye(din, dtype=dtype)], dim=0)*math.sqrt(din)
    test_x = torch.randn(256, din, generator=generator, dtype=dtype)
    train_x, test_x = train_x.to(device), test_x.to(device)
    with torch.no_grad():
        train_y = F.linear(train_x, base.weight+delta, base.bias)
        test_y = F.linear(test_x, base.weight+delta, base.bias)
    return {"base": base, "target_delta": delta,
            "train_x": train_x, "train_y": train_y, "test_x": test_x, "test_y": test_y}


def effective_weight(model):
    return model.base.weight + model.delta_weight() if isinstance(model, LoRALinear) else model.weight


def evaluate_model(model, x, y):
    mode = model.training
    try:
        model.eval()
        with torch.no_grad():
            return F.mse_loss(model(x), y).item()
    finally:
        model.train(mode)


def optimizer_tensor_bytes(optimizer):
    return sum(value.numel()*value.element_size()
               for state in optimizer.state.values() for value in state.values() if isinstance(value, torch.Tensor))


def fit_one(problem, label, rank, steps=400, lr=0.03):
    positive_int(steps, "steps")
    finite_positive(lr, "lr")
    if rank is None:
        model = copy.deepcopy(problem["base"])
        # 为隔离权重更新，所有路径都冻结bias；不是完整网络全参数微调。
        if model.bias is not None:
            model.bias.requires_grad_(False)
    else:
        # alpha=r，让各个rank的scaling均为1，而不是顺便改变缩放。
        model = LoRALinear(problem["base"], rank=rank, alpha=float(rank), dropout=0., seed=3215)
    optimizer = torch.optim.Adam(trainable_parameters(model), lr=lr, weight_decay=0.)
    base_before = state_sha256(cpu_state(problem["base"]))
    frozen_before = {n: p.detach().clone() for n, p in model.named_parameters() if not p.requires_grad}
    initial = evaluate_model(model, problem["train_x"], problem["train_y"])
    history = []
    model.train()
    for step in range(1, steps+1):
        optimizer.zero_grad(set_to_none=True)
        loss = F.mse_loss(model(problem["train_x"]), problem["train_y"])
        if not bool(torch.isfinite(loss)):
            raise RuntimeError(f"{label}: loss非有限")
        loss.backward()
        if any(p.grad is None or not bool(torch.isfinite(p.grad).all()) for p in trainable_parameters(model)):
            raise RuntimeError(f"{label}: 梯度缺失或非有限")
        optimizer.step()
        if step == 1 or step % 50 == 0 or step == steps:
            history.append({"step": step, "train_mse": evaluate_model(model, problem["train_x"], problem["train_y"])})
    model.eval()
    with torch.no_grad():
        learned_delta = effective_weight(model)-problem["base"].weight
        remaining = learned_delta-problem["target_delta"]
        sv = torch.linalg.svdvals(learned_delta.double())
    frozen_equal = all(torch.equal(dict(model.named_parameters())[n], p) for n, p in frozen_before.items())
    source_equal = base_before == state_sha256(cpu_state(problem["base"]))
    if not (frozen_equal and source_equal):
        raise RuntimeError("冻结参数或源基座被修改")
    report = {"label": label, "rank": rank, "alpha": float(rank) if rank else None,
              "scaling": 1.0 if rank else None, "steps": steps, "learning_rate": lr,
              "trainable": parameter_report(model)["trainable"], "total": parameter_report(model)["total"],
              "initial_train_mse": initial,
              "final_train_mse": evaluate_model(model, problem["train_x"], problem["train_y"]),
              "test_mse": evaluate_model(model, problem["test_x"], problem["test_y"]),
              "weight_error_frobenius": remaining.norm().item(),
              "learned_delta_singular_values": sv.cpu().tolist(),
              "frozen_parameters_unchanged": frozen_equal, "source_base_unchanged": source_equal,
              "base_gradient_none": all(p.grad is None for p in model.base.parameters()) if isinstance(model, LoRALinear) else None,
              "parameter_tensor_bytes": sum(p.numel()*p.element_size() for p in model.parameters()),
              "gradient_tensor_bytes": sum(p.grad.numel()*p.grad.element_size() for p in model.parameters() if p.grad is not None),
              "optimizer_state_tensor_bytes": optimizer_tensor_bytes(optimizer), "history": history}
    return model, report


def run_training(output_dir, device="cpu", steps=400, lr=0.03):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    problem = make_problem(device)
    reports, models = {}, {}
    frozen = copy.deepcopy(problem["base"]).requires_grad_(False).eval()
    reports["frozen"] = {"trainable": 0,
        "final_train_mse": evaluate_model(frozen, problem["train_x"], problem["train_y"]),
        "test_mse": evaluate_model(frozen, problem["test_x"], problem["test_y"])}
    for label, rank in [("full_weight", None), ("lora_r1", 1), ("lora_r2", 2), ("lora_r4", 4)]:
        model, report = fit_one(problem, label, rank, steps, lr)
        models[label], reports[label] = model, report
    # 保存精确基座；adapter不包含基座张量，恢复时必须配套。
    torch.save({"format": "lesson32-base-v1", "in_features":16, "out_features":12, "bias":True,
                "state": cpu_state(problem["base"])}, output_dir / "base.pt")
    save_adapter(models["lora_r2"], output_dir / "adapter_r2.pt")
    restored = load_adapter(problem["base"], output_dir / "adapter_r2.pt")
    merged = models["lora_r2"].merged_linear()
    torch.save({"format": "lesson32-merged-v1", "in_features":16,"out_features":12,"bias":True,
                "state":cpu_state(merged)}, output_dir / "merged_r2.pt")
    with torch.no_grad():
        expected = models["lora_r2"](problem["test_x"])
        restore_error = (expected-restored(problem["test_x"])).abs().max().item()
        merge_error = (expected-merged(problem["test_x"])).abs().max().item()
    payload = torch.load(output_dir / "adapter_r2.pt", map_location="cpu", weights_only=True)
    metadata = {"task": "固定随机基座上的已知rank=2线性教师；不是语言模型能力比较",
                "dtype": str(problem["train_x"].dtype), "device": str(device),
                "train_samples": 32, "test_samples":256, "in_features":16,"out_features":12,
                "target_singular_values": [2.,1.],
                "theoretical_train_floor": {"frozen":5/12, "lora_r1":1/12, "lora_r2":0.0, "lora_r4":0.0, "full_weight":0.0},
                "selection": "固定步数，不根据测试集选择rank/超参数/最佳轮次",
                "restore_error":restore_error, "merge_error":merge_error,
                "adapter_keys":list(payload["state"]),
                "base_fp32_tensor_bytes":sum(t.numel()*t.element_size() for t in cpu_state(problem["base"]).values()),
                "adapter_fp32_tensor_bytes":sum(t.numel()*t.element_size() for t in payload["state"].values()),
                "base_file_bytes":(output_dir / "base.pt").stat().st_size,
                "adapter_file_bytes":(output_dir / "adapter_r2.pt").stat().st_size,
                "note": "小文件头可能大于张量；元素载荷不等于.pt大小或显存峰值。"}
    result = {"protocol": metadata, "branches": reports}
    (output_dir / "training.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    return result

In [18]:
problem = make_problem(dtype=torch.float64)
second_moment = problem["train_x"].T @ problem["train_x"] / problem["train_x"].shape[0]
print("训练/测试样本数：", len(problem["train_x"]), len(problem["test_x"]))
print("输入二阶矩与I的最大误差：", (second_moment-torch.eye(16,dtype=torch.float64)).abs().max().item())
print("目标更新前四个奇异值：", torch.linalg.svdvals(problem["target_delta"])[:4])

训练/测试样本数： 32 256
输入二阶矩与I的最大误差： 0.0
目标更新前四个奇异值： tensor([2.0000e+00, 1.0000e+00, 1.6660e-16, 9.0370e-17], dtype=torch.float64)


### 8.1 为什么r=1的训练MSE不可能达到零？

对本课特别构造的训练输入，有：

$$\mathrm{MSE}=\frac{1}{12}\|\Delta W-\Delta W_*\|_F^2$$

目标增量有两个正交方向，强度分别为2和1。rank=1最多完整保留最强的一个；最小残差平方为 $1^2$，因此最低MSE为：

$$L_{r=1,\min}=\frac{1}{12}\approx0.0833333$$

不适配时MSE为 $(2^2+1^2)/12=5/12$。r≥2时这个目标理论上可精确表达，但**优化器并非在任何设置下都保证达到理论下界**。

这是一条单层线性任务的计算结论，不能直接推导真实语言模型的最佳r。讲义用SVD构造rank=1最优解，与训练结果进行核对；SVD用于分析答案，不用于初始化LoRA。

In [19]:
u, singular, vh = torch.linalg.svd(problem["target_delta"], full_matrices=False)
best_rank1 = (u[:, :1] * singular[:1]) @ vh[:1]
oracle = copy.deepcopy(problem["base"])
with torch.no_grad():
    oracle.weight.add_(best_rank1)
print("不更新：", evaluate_model(problem["base"],problem["train_x"],problem["train_y"]))
print("rank1解析最优：", evaluate_model(oracle,problem["train_x"],problem["train_y"]))
print("理论rank1下界：", 1/12)

不更新： 0.41666666666666674
rank1解析最优： 0.08333333333333336
理论rank1下界： 0.08333333333333333


### 8.2 对照协议

比较五条路径：冻结不动、全量更新单个W、LoRA r=1、r=2、r=4。所有路径共享W0与数据，偏置固定。可训练分支使用Adam、400次全批更新、学习率0.03、weight_decay=0、Dropout=0。

这里的 `full_weight` 指**整个单层权重W可训练**，不是完整语言模型全参数微调。LoRA统一alpha=r。A/B不能在不同rank下完全相同形状，结果也不用于宣称哪个方法收敛速度最优。

运行会保存 `base.pt`、`adapter_r2.pt`、`merged_r2.pt` 及指标。合并权重为独立文件；不修改原adapter或基座。

In [20]:
training_result = None
if RUN_TRAINING:
    training_result = run_training(OUTPUT_DIR, device=device, steps=STEPS, lr=LR)
    print("完整实验已经保存到：", OUTPUT_DIR.resolve())
else:
    print("完整训练已跳过；前面的数学与梯度实验仍可学习。")

完整实验已经保存到： /mnt/data/_lesson32_work/notebook_run/lesson_32_outputs


In [21]:
if training_result is not None:
    print(f"{'分支':12s} {'可训练数':>8s} {'训练MSE':>15s} {'测试MSE':>15s}")
    for name, row in training_result["branches"].items():
        print(f"{name:12s} {row['trainable']:8d} {row['final_train_mse']:15.7e} {row['test_mse']:15.7e}")
    print("adapter重载误差：", training_result["protocol"]["restore_error"])
    print("合并误差：", training_result["protocol"]["merge_error"])

分支               可训练数           训练MSE           测试MSE
frozen              0   4.1666666e-01   3.8832235e-01
full_weight       192   2.8826767e-15   5.1128027e-15
lora_r1            28   8.3333328e-02   7.6837569e-02
lora_r2            56   2.4283960e-15   9.6470480e-15
lora_r4           112   1.8711523e-15   8.6780829e-15
adapter重载误差： 0.0
合并误差： 4.76837158203125e-07


### 8.3 怎样解释结果，不夸大结论

先检查r=1是否接近1/12，再看r≥2是否显著降低误差。检查记录中的 `frozen_parameters_unchanged` 与 `source_base_unchanged`，确认没有通过偷偷改变W0来拟合。

测试输入虽然独立，但来自同一个人为规定的线性映射，没有语言理解、编程、指令遵循或灾难性遗忘评测。小误差也不等于参数逐位恢复：A/B本身不是唯一分解。

例如，任意可逆的r×r矩阵C都可以给出另一组分解：

$$(BC)(C^{-1}A)=BA$$

所以，应比较**有效ΔW或模型输出**，而不是要求训练得到的A、B等于某一份教师分解。

In [22]:
if training_result is not None:
    for name, row in training_result["branches"].items():
        if name == "frozen":
            continue
        print(name, "基座/冻结参数不变：", row["source_base_unchanged"], row["frozen_parameters_unchanged"],
              "参数/梯度/Adam张量字节：", row["parameter_tensor_bytes"], row["gradient_tensor_bytes"],
              row["optimizer_state_tensor_bytes"])
    print("文件与元素载荷：", {k:v for k,v in training_result["protocol"].items() if k.endswith("bytes")})

full_weight 基座/冻结参数不变： True True 参数/梯度/Adam张量字节： 816 768 1540
lora_r1 基座/冻结参数不变： True True 参数/梯度/Adam张量字节： 928 112 232
lora_r2 基座/冻结参数不变： True True 参数/梯度/Adam张量字节： 1040 224 456
lora_r4 基座/冻结参数不变： True True 参数/梯度/Adam张量字节： 1264 448 904
文件与元素载荷： {'base_fp32_tensor_bytes': 816, 'adapter_fp32_tensor_bytes': 224, 'base_file_bytes': 2637, 'adapter_file_bytes': 2429}


## 9. LoRA 小在哪里？仍然大在哪里？

本课r=2的A/B共56个FP32数，载荷224字节；基座权重和偏置共204个数，载荷816字节。这不是 `.pt` 文件大小：小文件的元数据/容器开销可能超过张量本身。

LoRA主要减少可训练梯度和优化器状态，以及每个任务需要单独保存的增量。基座仍占空间，反传路径和激活也可能保留，因此不能把可训练参数比例等同于峰值显存比例。[1][4]

**QLoRA** 则把冻结的量化基座与可训练的LoRA结合起来，还要处理量化模块、计算dtype与训练准备。本课没有量化，不能因为有LoRA就叫QLoRA。[7]

本课没有测运行速度或GPU峰值显存。没有CUDA时，不会给3060填写模拟节省比例。

In [23]:
if training_result is not None:
    reload_report = reload_saved_run(OUTPUT_DIR, device=device)
    print("完全从磁盘重建：")
    print(json.dumps(reload_report, ensure_ascii=False, indent=2))

完全从磁盘重建：
{
  "device": "cpu",
  "adapter_trainable": 56,
  "base_sha256": "39e4e1a8114b327ce72e9bbe499f775384b1bcfd6f83fb94c815afde13a72947",
  "adapter_vs_merged_error": 1.1920928955078125e-07,
  "note": "仅恢复推理；没有优化器、数据游标或RNG，不能精确续训"
}


## 10. 自动检查：不只检查loss有没有下降

下面的固定检查覆盖：矩阵方向、缩放、二维/三维与非连续输入、输出和梯度、初始零增量、两矩阵全零陷阱、冻结路径、Dropout、目标注入、合并、adapter协议与秩下界。

检查使用FP64验证数学等价性。训练默认FP32，所以合并与分支路径允许小的浮点误差；不要求不同硬件/版本逐位一致。

In [24]:
# %% 07. Fixed checks, counted only after each named check passes
class Checks:
    def __init__(self):
        self.names = []

    def ok(self, name, condition):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def close(self, name, actual, expected, rtol=1e-8, atol=1e-10):
        try:
            torch.testing.assert_close(actual, expected, rtol=rtol, atol=atol)
        except AssertionError as error:
            raise AssertionError(name) from error
        self.names.append(name)

    def raises(self, name, fn, kind=(ValueError, TypeError, RuntimeError)):
        try:
            fn()
        except kind:
            self.names.append(name)
        else:
            raise AssertionError(name + "：预期拒绝却成功")


def run_checks():
    c = Checks()
    base = seeded_linear(5, 4, dtype=torch.float64)
    l = LoRALinear(base, 2, 6.0)
    c.ok("A形状", tuple(l.lora_A.shape)==(2,5))
    c.ok("B形状", tuple(l.lora_B.shape)==(4,2))
    c.ok("alpha/r", l.scaling==3.0)
    c.ok("A非零", l.lora_A.count_nonzero()>0)
    c.ok("B全零", l.lora_B.count_nonzero()==0)
    c.ok("基座冻结", all(not p.requires_grad for p in l.base.parameters()))
    c.ok("只有A/B可训练", {n for n,p in l.named_parameters() if p.requires_grad}=={"lora_A","lora_B"})
    c.ok("源基座未被冻结", all(p.requires_grad for p in base.parameters()))
    c.ok("基座无参数别名", base.weight.data_ptr()!=l.base.weight.data_ptr())
    c.ok("LoRA参数数", parameter_report(l)["trainable"]==2*(5+4))
    x = torch.arange(30, dtype=torch.float64).reshape(2,3,5)/10
    c.close("初始输出等于基座", l(x), base(x), rtol=0,atol=0)
    c.ok("三维输入输出形状", tuple(l(x).shape)==(2,3,4))
    c.ok("一维输入", tuple(l(x[0,0]).shape)==(4,))
    c.ok("无偏置层", LoRALinear(seeded_linear(5,4,bias=False,dtype=torch.float64)).base.bias is None)
    g = torch.Generator().manual_seed(333)
    with torch.no_grad():
        l.lora_B.copy_(torch.randn(4,2,generator=g,dtype=torch.float64))
    l.eval()
    dense = base.weight+l.scaling*(l.lora_B@l.lora_A)
    c.close("分支与稠密公式", l(x), F.linear(x,dense,base.bias))
    noncontiguous = torch.arange(60,dtype=torch.float64).reshape(2,3,10)[...,::2]
    c.close("非连续输入", l(noncontiguous), F.linear(noncontiguous,dense,base.bias))
    c.ok("ΔW秩上界", torch.linalg.matrix_rank(l.delta_weight()).item()<=2)
    probe=x.clone().requires_grad_(True)
    actual= l(probe).square().sum()
    actual_grads=torch.autograd.grad(actual,(probe,l.lora_A,l.lora_B))
    refx=x.clone().requires_grad_(True)
    a=l.lora_A.detach().clone().requires_grad_();b=l.lora_B.detach().clone().requires_grad_()
    ref=F.linear(refx,base.weight+l.scaling*(b@a),base.bias).square().sum()
    refgrads=torch.autograd.grad(ref,(refx,a,b))
    for name, actual_g,ref_g in zip(["输入","A","B"],actual_grads,refgrads):
        c.close("稠密梯度核对"+name,actual_g,ref_g)
    # 本测试故意使alpha/r != 1，能发现漏缩放。
    wrong=base(x)+F.linear(F.linear(x,l.lora_A),l.lora_B)
    c.ok("漏缩放有差异", not torch.allclose(l(x),wrong))
    init=initialization_demo()
    c.ok("首次A梯度为零", init["steps"][0]["A_grad_norm"]==0)
    c.ok("首次B梯度非零", init["steps"][0]["B_grad_norm"]>0)
    c.ok("第二次A梯度非零", init["steps"][1]["A_grad_norm"]>0)
    c.ok("基座grad为None", all(s["base_grad_none"] for s in init["steps"]))
    c.ok("两矩阵全零会卡住", init["both_zero_A_grad_norm"]==init["both_zero_B_grad_norm"]==0)
    fp=frozen_path_demo()
    c.ok("穿过冻结层的梯度", fp["upstream_A_grad_norm"]>0)
    c.ok("冻结层自身无梯度", fp["frozen_output_grad_none"])
    c.ok("no_grad错误路径断开", fp["wrong_path_cannot_backpropagate"])
    # Dropout只作用于增量分支。
    drop=LoRALinear(base,2,6.,dropout=.5)
    c.close("B零时训练dropout不改基座输出",drop(x),base(x),rtol=0,atol=0)
    with torch.no_grad(): drop.lora_B.fill_(.2)
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(322)
        got=drop(x)
        torch.manual_seed(322)
        expect=base(x)+drop.scaling*F.linear(F.linear(F.dropout(x,.5,True),drop.lora_A),drop.lora_B)
    c.close("只对LoRA输入Dropout",got,expect)
    drop.eval()
    c.close("eval关闭Dropout",drop(x),F.linear(x,base.weight+drop.delta_weight(),base.bias))
    c.raises("训练模式拒绝合并",lambda:drop.train().merged_linear())
    drop.eval()
    before=state_sha256(cpu_state(drop)); merged=drop.merged_linear()
    c.close("合并前后等价",merged(x),drop(x))
    c.ok("合并不修改源",before==state_sha256(cpu_state(drop)))
    c.ok("导出独立权重",merged.weight.data_ptr()!=drop.base.weight.data_ptr())
    c.ok("合并结果没有LoRA",type(merged) is nn.Linear and not any("lora" in n for n,_ in merged.named_parameters()))
    c.close("多次导出不累加",merged(x),drop.merged_linear()(x),rtol=0,atol=0)
    c.ok("双算会改变输出",merge_demo()["double_count_error"]>0.1)
    # 输入验证。
    for value in [0,-1,True,1.5,6]:
        c.raises(f"拒绝rank={value!r}",lambda v=value:LoRALinear(base,v))
    for value in [0.,-1.,float("nan"),float("inf"),True]:
        c.raises(f"拒绝alpha={value!r}",lambda v=value:LoRALinear(base,2,v))
    for value in [-.1,1.,float("nan"),True]:
        c.raises(f"拒绝dropout={value!r}",lambda v=value:LoRALinear(base,2,2.,v))
    c.raises("拒绝非Linear",lambda:LoRALinear(nn.ReLU()))
    c.raises("拒绝FP16教学基座",lambda:LoRALinear(copy.deepcopy(base).half()))
    c.raises("输入末维错误",lambda:l(torch.ones(2,6,dtype=torch.float64)))
    # 注入与可训练参数审计。
    network=TinyTwoLayer();snapshot=state_sha256(cpu_state(network))
    adapted=inject_lora(network,["proj_in","proj_out"],2,4.)
    xx=torch.randn(3,6,generator=g,dtype=torch.float64)
    c.close("注入初始输出不变",network(xx),adapted(xx),rtol=0,atol=0)
    c.ok("源网络不改写",state_sha256(cpu_state(network))==snapshot)
    c.ok("两层都命中",sum(isinstance(m,LoRALinear) for m in adapted.modules())==2)
    c.ok("注入参数计数",parameter_report(adapted)["trainable"]==2*(6+8)+2*(8+4))
    F.mse_loss(adapted(xx),torch.ones(3,4,dtype=torch.float64)).backward()
    c.ok("每层B收到梯度",all(m.lora_B.grad is not None and m.lora_B.grad.norm()>0 for m in adapted.modules() if isinstance(m,LoRALinear)))
    c.raises("目标名写错不能静默",lambda:inject_lora(network,["missing"]))
    c.raises("目标重名拒绝",lambda:inject_lora(network,["proj_in","proj_in"]))
    c.raises("空目标拒绝",lambda:inject_lora(network,[]))
    c.raises("非Linear目标拒绝",lambda:inject_lora(nn.Sequential(nn.ReLU()),["0"]))
    c.raises("重复注入拒绝",lambda:inject_lora(adapted,["proj_in"]))
    shared=nn.ModuleList([network.proj_in,network.proj_in])
    c.raises("绑定模块拒绝",lambda:inject_lora(shared,["0"]))
    sequential=inject_lora(nn.Sequential(seeded_linear(5,4,dtype=torch.float64)),["0"],2,4.)
    c.ok("完整数字路径可注入",isinstance(sequential[0],LoRALinear))
    # adapter协议与文件。
    payload=adapter_payload(l)
    c.ok("只保存A/B",set(payload["state"])=={"lora_A","lora_B"})
    c.ok("保存缩放元信息",payload["config"]["alpha"]==6. and payload["config"]["rank"]==2)
    restored=adapter_from_payload(base,payload)
    c.close("payload重建",restored(x),l(x),rtol=0,atol=0)
    with torch.no_grad(): l.lora_A.add_(1.)
    c.ok("保存payload是快照",not torch.equal(payload["state"]["lora_A"],l.lora_A.cpu()))
    wrongbase=copy.deepcopy(base)
    with torch.no_grad():wrongbase.weight[0,0]+=0.01
    c.raises("同形状错误基座拒绝",lambda:adapter_from_payload(wrongbase,payload))
    for key,value in [("format","wrong"),("base_sha256","x")]:
        bad=copy.deepcopy(payload);bad[key]=value
        c.raises("协议拒绝"+key,lambda b=bad:adapter_from_payload(base,b))
    bad=copy.deepcopy(payload);bad["config"]["alpha"]=0
    c.raises("元数据非法alpha",lambda:adapter_from_payload(base,bad))
    bad=copy.deepcopy(payload);bad["state"]["lora_B"]=torch.zeros(4,3,dtype=torch.float64)
    c.raises("错误adapter形状",lambda:adapter_from_payload(base,bad))
    bad=copy.deepcopy(payload);bad["state"]["lora_B"][0,0]=float("nan")
    c.raises("非有限adapter拒绝",lambda:adapter_from_payload(base,bad))
    with tempfile.TemporaryDirectory(prefix="lesson32-check-") as tmp:
        path=Path(tmp)/"adapter.pt";save_adapter(restored,path);loaded=load_adapter(base,path)
        c.close("磁盘重载输出",loaded(x),restored(x),rtol=0,atol=0)
        c.ok("加载不改变基座",state_sha256(cpu_state(base))==payload["base_sha256"])
    # 可解析的秩限制实验。
    problem=make_problem(dtype=torch.float64)
    covariance=problem["train_x"].T@problem["train_x"]/problem["train_x"].shape[0]
    c.close("训练二阶矩为I",covariance,torch.eye(16,dtype=torch.float64))
    c.ok("目标更新rank2",torch.linalg.matrix_rank(problem["target_delta"]).item()==2)
    c.close("冻结目标MSE",torch.tensor(evaluate_model(problem["base"],problem["train_x"],problem["train_y"])),torch.tensor(5/12),rtol=1e-6,atol=1e-7)
    u,s,vh=torch.linalg.svd(problem["target_delta"],full_matrices=False)
    rank1=(u[:,:1]*s[:1])@vh[:1]
    oracle=copy.deepcopy(problem["base"])
    with torch.no_grad():oracle.weight.add_(rank1)
    c.close("rank1最优MSE",torch.tensor(evaluate_model(oracle,problem["train_x"],problem["train_y"])),torch.tensor(1/12),rtol=1e-6,atol=1e-7)
    counts=capacity_demo()
    c.ok("大投影参数比",counts["lora_parameters"]==65536 and counts["full_weight_parameters"]==16777216)
    return {"passed":len(c.names),"names":c.names,"device":"cpu","dtype":"float64 where applicable"}

In [25]:
check_report = run_checks()
print(f"固定检查通过：{check_report['passed']} 项")
print("部分检查名：", check_report["names"][:8])

固定检查通过：85 项
部分检查名： ['A形状', 'B形状', 'alpha/r', 'A非零', 'B全零', '基座冻结', '只有A/B可训练', '源基座未被冻结']


## 11. 独立练习：先完成，再打开参考答案

下面的检查开关默认关闭，函数中的 `NotImplementedError` 提醒你尚未实现。**运行所有单元成功，不代表练习完成。**

### 练习A：手写无Dropout的LoRA前向

实现 `lora_forward_ex(x, weight, bias, A, B, alpha)`：

- 权重方向 `[out,in]`，A `[r,in]`，B `[out,r]`。
- 输入可以是 `[B,T,in]`，bias可以为None。
- 返回仍能反向传播的输出，不用detach或no_grad。
- 使用 `alpha/r`，检查器特意使用不等于1的缩放。

In [26]:
def lora_forward_ex(x, weight, bias, A, B, alpha):
    raise NotImplementedError("请实现两个线性路径与alpha/r缩放")


def adapter_step_ex(layer, optimizer, x, target):
    raise NotImplementedError("请实现一次只更新适配器的训练步骤")


def merge_ex(layer):
    raise NotImplementedError("请在eval下返回独立的合并nn.Linear")

RUN_EXERCISE_A = False
RUN_EXERCISE_B = False
RUN_EXERCISE_C = False

In [27]:
def check_exercise_a(fn: Callable):
    g=torch.Generator().manual_seed(3222)
    vals=[torch.randn(*s,generator=g,dtype=torch.float64).requires_grad_()
          for s in [(2,3,5),(4,5),(4,),(2,5),(4,2)]]
    x,w,b,a,bb=vals
    got=fn(x,w,b,a,bb,6.0)
    expected=F.linear(x,w+3*(bb@a),b)
    torch.testing.assert_close(got,expected)
    grads=torch.autograd.grad(got.square().sum(),vals,retain_graph=True)
    refgrads=torch.autograd.grad(expected.square().sum(),vals)
    for grad,ref in zip(grads,refgrads):torch.testing.assert_close(grad,ref)
    torch.testing.assert_close(fn(x,w,None,a,bb,6.),F.linear(x,w+3*(bb@a),None))
    return "练习A：输出、三维输入、无偏置及所有输入梯度通过"


def check_exercise_b(fn: Callable):
    base=seeded_linear(5,4,dtype=torch.float64)
    model=LoRALinear(base,2,4.)
    reference=copy.deepcopy(model)
    opt=torch.optim.SGD(trainable_parameters(model),lr=.1)
    refopt=torch.optim.SGD(trainable_parameters(reference),lr=.1)
    x=torch.arange(20,dtype=torch.float64).reshape(4,5)/20
    target=torch.ones(4,4,dtype=torch.float64)
    before=state_sha256(cpu_state(model.base))
    # 先填旧梯度，验证实现确实清零；两次更新验证A也能动起来。
    for p in trainable_parameters(model):p.grad=torch.ones_like(p)*123
    for _ in range(2):
        loss=fn(model,opt,x,target)
        refopt.zero_grad(set_to_none=True)
        ref_loss=F.mse_loss(reference(x),target)
        ref_loss.backward()
        refopt.step()
        expected=float(ref_loss.detach())
        if not isinstance(loss,float) or not math.isclose(loss,expected,rel_tol=1e-10,abs_tol=1e-10):
            raise AssertionError("返回值必须是本次更新前的浮点MSE")
    for n,p in model.named_parameters():torch.testing.assert_close(p,dict(reference.named_parameters())[n])
    if state_sha256(cpu_state(model.base))!=before or any(p.grad is not None for p in model.base.parameters()):
        raise AssertionError("基座被改动或出现梯度")
    return "练习B：两次更新、旧梯度清零、基座不变通过"


def check_exercise_c(fn: Callable):
    l=LoRALinear(seeded_linear(5,4,dtype=torch.float64),2,6.,.2).eval()
    with torch.no_grad():l.lora_B.fill_(.2)
    x=torch.arange(30,dtype=torch.float64).reshape(2,3,5)/10
    before=state_sha256(cpu_state(l))
    got=fn(l)
    if type(got) is not nn.Linear or got.weight.data_ptr()==l.base.weight.data_ptr():
        raise AssertionError("必须返回独立nn.Linear")
    torch.testing.assert_close(got(x),l(x))
    if before!=state_sha256(cpu_state(l)):raise AssertionError("不能改写源模型")
    again=fn(l)
    torch.testing.assert_close(again(x),got(x))
    try: fn(l.train())
    except (ValueError,RuntimeError):pass
    else:raise AssertionError("训练模式应拒绝合并")
    return "练习C：eval合并等价、重复导出、不改源、拒绝train通过"

In [28]:
if RUN_EXERCISE_A:
    print(check_exercise_a(lora_forward_ex))
else:
    print("练习A待完成；补全后将RUN_EXERCISE_A改为True。")

练习A待完成；补全后将RUN_EXERCISE_A改为True。


### 练习B：实现一次adapter更新

传入的layer已冻结基座，optimizer已指向A/B。你需要切换train模式、清旧梯度、计算均方误差、backward、step，返回**更新前的loss浮点值**。

检查器先填入旧梯度，再连续调用两次，验证确实清零。不能直接用 `.data` 手动覆盖A/B，也不能在no_grad里计算loss。

In [29]:
if RUN_EXERCISE_B:
    print(check_exercise_b(adapter_step_ex))
else:
    print("练习B待完成；检查器会比较连续两次参数更新。")

练习B待完成；检查器会比较连续两次参数更新。


### 练习C：导出合并副本

接收一个 `LoRALinear`，返回独立普通Linear。要求：

- 训练模式明确报错；只合并eval模型。
- 权重包含W0和一次缩放后的BA，偏置不重复添加。
- 不修改源LoRA、不共享参数存储、不保留adapter分支。
- 连续导出两次仍得到相同结果。

不要直接调用现成的 `layer.merged_linear()`；用本课的等式自己实现。

In [30]:
if RUN_EXERCISE_C:
    print(check_exercise_c(merge_ex))
else:
    print("练习C待完成；重点防止原地改写与增量重复计入。")

练习C待完成；重点防止原地改写与增量重复计入。


---
## 12. 参考答案

先暂停，在空白单元完成练习，再展开。参考答案采用独立名字，不覆盖学生函数。

In [31]:
def reference_lora_forward(x, weight, bias, A, B, alpha):
    return F.linear(x, weight, bias)+(float(alpha)/A.shape[0])*F.linear(F.linear(x,A),B)


def reference_adapter_step(layer, optimizer, x, target):
    layer.train()
    optimizer.zero_grad(set_to_none=True)
    loss=F.mse_loss(layer(x),target)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def reference_merge(layer):
    if layer.training:
        raise RuntimeError("需要eval模式")
    result=copy.deepcopy(layer.base)
    with torch.no_grad():
        result.weight.copy_(layer.base.weight+(layer.alpha/layer.rank)*(layer.lora_B@layer.lora_A))
    result.requires_grad_(False)
    return result.eval()

In [32]:
print(check_exercise_a(reference_lora_forward))
print(check_exercise_b(reference_adapter_step))
print(check_exercise_c(reference_merge))

练习A：输出、三维输入、无偏置及所有输入梯度通过
练习B：两次更新、旧梯度清零、基座不变通过
练习C：eval合并等价、重复导出、不改源、拒绝train通过


## 13. 易错点复盘

| 容易误解的说法 | 本课结论 |
|---|---|
| LoRA把原权重变成低秩了 | 低秩约束施加在增量，W0仍然保留 |
| A首步梯度为零就是失败 | B=0初始化时这是链式法则的结果 |
| 两个矩阵都零最安全 | 两边的梯度都可能锁为零 |
| 冻结基座就可以整个no_grad | 更早的adapter可能需要梯度穿过冻结层 |
| r越大一定更好 | 资源增加，任务/优化/泛化仍需评测 |
| alpha就是学习率 | 前向缩放与优化器步长是不同机制 |
| adapter文件单独就是模型 | 还要正确基座与匹配配置 |
| merge之后再加adapter | 增量重复计算，输出错误 |
| LoRA与QLoRA相同 | 后者还涉及冻结量化基座与对应训练路径 |

验收：能解释初始化两步梯度、用两种路径计算同一输出、从磁盘恢复正确基座+adapter，并拒绝同形状错误基座。

## 14. 脚本命令与输出说明

在文件所在目录：

```bash
python lesson_32_lora_from_scratch.py --checks-only
python lesson_32_lora_from_scratch.py --math-only
python lesson_32_lora_from_scratch.py
```

可用CUDA环境中，可以添加 `--device cuda`；显式CUDA不可用会报错，`--device auto`会选择实际可用设备。几百参数的实验无须显卡，这里不是速度基准。

已有文件时，开一个新进程验证独立恢复，不重新训练：

```bash
python lesson_32_lora_from_scratch.py --reload-dir lesson_32_outputs
```

默认生成 `base.pt`、`adapter_r2.pt`、`merged_r2.pt`、`mechanisms.json`、`training.json`、`experiment.json`。同名文件会覆盖。

**这些不是完整断点续训检查点**：没有优化器、数据游标、随机状态。格式也不兼容PEFT；只接受可信的本课文件。

下一课：**使用PEFT给真实小模型注入LoRA**，核对模块命中、参数冻结、初始输出和梯度，再连接监督微调数据与回归评测。

In [33]:
# 写入本Notebook的汇总，与脚本使用同一实现/相同协议。
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
math_report = {"geometry": geometry_demo(), "initialization": initialization_demo(),
               "frozen_path": frozen_path_demo(), "merge": merge_demo(), "capacity": capacity_demo()}
(OUTPUT_DIR/"mechanisms.json").write_text(json.dumps(math_report,ensure_ascii=False,indent=2),encoding="utf-8")
experiment = {"lesson":32,"python":platform.python_version(),"torch":str(torch.__version__),
              "device":str(device),"cuda_available":torch.cuda.is_available(),"checks":check_report,
              "mechanisms":math_report,"training":training_result,"sources":SOURCES,
              "limits":["合成线性教师，不是语言/推理/编程能力评测", "未测峰值显存或速度", "dense FP32/FP64，不是QLoRA",
                        "adapter教学格式，不兼容PEFT；没有完整断点续训状态"]}
(OUTPUT_DIR/"experiment.json").write_text(json.dumps(experiment,ensure_ascii=False,indent=2),encoding="utf-8")
print("报告已写入：",OUTPUT_DIR.resolve())

报告已写入： /mnt/data/_lesson32_work/notebook_run/lesson_32_outputs


## 15. 一手资料与本课边界

1. [Hu et al., LoRA 原论文](https://arxiv.org/html/2106.09685v2)：低秩更新、冻结基座、初始化、缩放与合并。
2. [Hugging Face PEFT：LoRA](https://huggingface.co/docs/peft/package_reference/lora)：初始化、target_modules、变体与合并接口。在线页面可能变化；本课未安装/测试PEFT，不依赖其接口执行。
3. [PyTorch Linear](https://docs.pytorch.org/docs/2.10/generated/torch.nn.Linear.html)：权重方向、输入末维和偏置。
4. [PyTorch Autograd mechanics](https://docs.pytorch.org/docs/2.10/notes/autograd.html)：requires_grad、no_grad与冻结。
5. [PEFT checkpoint format](https://huggingface.co/docs/peft/developer_guides/checkpoint)：adapter参数、配置与基座依赖。
6. [PyTorch Serialization](https://docs.pytorch.org/docs/2.10/notes/serialization.html)：加载容器与weights_only边界。
7. [QLoRA 原论文](https://arxiv.org/abs/2305.14314)：冻结四位基座与可训练低秩适配器。
8. [Microsoft LoRA 原始实现](https://github.com/microsoft/LoRA/blob/main/loralib/layers.py)：Linear矩阵方向与初始化实现参考。

本课代码、数值与训练结果来自实际CPU运行。没有真实语言模型微调、CUDA测量或任务能力结论。可选CUDA路径需要在你的3060环境验证。